# Figures 5.3-5.5 - epoch-budget sweep (colab42, arm E)

Regenerates the three epoch-budget line graphs used in thesis section 5.2.3.

**This notebook computes nothing.** Every value below is transcribed from
`colab_outputs/colab42_ablations_mean.csv`, rows `arm == "E-epochs"` - the
ablation run of record (2026-08-28, 180 rows, 15 configurations x 3 seeds x 4
datasets, evaluation fingerprint `7469de2003d4f7dc`). The numbers are embedded
so the notebook runs standalone in Colab with no repository checkout and no
Drive mount.

Design decisions, fixed and not to be changed without asking:

* three separate figures, never a shared y-axis - AUROC varies within a much
  narrower interval than Spearman or MAP@10, so a common scale would flatten it
* x-axis is the epoch budget on a **proportional numeric** scale, not categorical
* markers at every checkpoint, no smoothing
* epoch 30 marked with a thin dashed vertical line labelled "deployed"
* **no error bars** - they were shown once and cut on 2026-08-30 as too
  cluttered, and the rest of the document reports means with no variability
* palette is Chapter 4's and must not change; the AA/SS pair sits at dE 7.8
  under protanopia, inside the band that is only legal with a second encoding
  channel, so every dataset also carries its own **marker shape**

The Spearman panel spans 0.15-1.00 because AA sits far below the other three;
the inset magnifies the 0.90-0.98 band so their shapes stay readable.


In [ ]:
# =====================================================================
#  Values transcribed from colab42_ablations_mean.csv, arm == "E-epochs".
#  Order in every list is epochs = [5, 10, 20, 30, 50].
#  Verified against the CSV on 2026-08-30.
# =====================================================================
EPOCHS = [5, 10, 20, 30, 50]
DEPLOYED = 30

VALUES = {
    "spearman": {
        "Synth": [0.931242, 0.930181, 0.926533, 0.925128, 0.920857],
        "3Di":   [0.948251, 0.951989, 0.955616, 0.954348, 0.955272],
        "SS":    [0.954214, 0.959784, 0.961757, 0.963647, 0.960288],
        "AA":    [0.280711, 0.209074, 0.205961, 0.200759, 0.171395],
    },
    "auroc": {
        "Synth": [0.970823, 0.970377, 0.969420, 0.968201, 0.966279],
        "3Di":   [0.990073, 0.990310, 0.991352, 0.991621, 0.991875],
        "SS":    [0.976467, 0.982084, 0.984844, 0.986606, 0.986281],
        "AA":    [0.999945, 0.999119, 0.999449, 0.999449, 0.998734],
    },
    "map10": {
        "Synth": [0.989034, 0.987787, 0.979215, 0.973706, 0.964155],
        "3Di":   [0.507892, 0.509191, 0.507793, 0.508379, 0.509535],
        "SS":    [0.413371, 0.413787, 0.416322, 0.413392, 0.409506],
        "AA":    [0.897222, 0.953333, 0.908333, 0.883333, 0.867778],
    },
}

for _metric, _rows in VALUES.items():
    assert set(_rows) == {"Synth", "3Di", "SS", "AA"}, _metric
    for _d, _v in _rows.items():
        assert len(_v) == len(EPOCHS), (_metric, _d)
print("data OK:", {m: len(r) for m, r in VALUES.items()})


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Chapter 4's palette. Do not change - it is used by every dataset-coloured
# figure in the thesis. Marker shapes are the required secondary encoding.
COLORS  = {"Synth": "#FF7F0E", "3Di": "#0072B2", "SS": "#D62728", "AA": "#4D4D4D"}
MARKERS = {"Synth": "o",       "3Di": "s",       "SS": "^",       "AA": "D"}
ORDER   = ["Synth", "3Di", "SS", "AA"]

METRICS = [
    ("spearman", "Spearman rank correlation", (0.15, 1.00),  "colab42_epochs_spearman.png"),
    # 0.964, not 0.15-1.00: every AUROC value sits above 0.966, so a full-range
    # axis would render all four series as one flat line.
    ("auroc",    "AUROC",                     (0.964, 1.002), "colab42_epochs_auroc.png"),
    ("map10",    "MAP@10",                    (0.38, 1.01),  "colab42_epochs_map10.png"),
]
print("config OK")


In [ ]:
def draw(ax, metric, datasets, legend=True, small=False):
    for name in datasets:
        ax.plot(
            EPOCHS, VALUES[metric][name],
            color=COLORS[name], marker=MARKERS[name],
            markersize=4.5 if small else 6, linewidth=1.6 if small else 2.0,
            label=name if legend else None, zorder=3,
        )


def build(metric, ylabel, ylim, fname):
    fig, ax = plt.subplots(figsize=(6.0, 3.7))

    ax.axvline(DEPLOYED, color="#666666", linestyle="--", linewidth=0.9, zorder=1)
    # Label sits ABOVE the axes: inside, it collided with AA on the Spearman
    # panel and with SS on MAP@10.
    ax.annotate("deployed", xy=(DEPLOYED, 1.0), xycoords=("data", "axes fraction"),
                xytext=(0, 3), textcoords="offset points",
                fontsize=8, color="#666666", ha="center", va="bottom")

    draw(ax, metric, ORDER)

    ax.set_xlabel("training epochs")
    ax.set_ylabel(ylabel)
    ax.set_ylim(*ylim)
    ax.set_xlim(2, 53)
    ax.set_xticks(EPOCHS)
    ax.set_xticklabels([str(e) for e in EPOCHS])
    ax.grid(True, linewidth=0.4, alpha=0.35, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    if metric == "spearman":
        # Band marker rather than mark_inset connectors: the magnified band is
        # at the TOP of the axes and the inset sits mid-right, so the connector
        # lines ran diagonally across the whole figure.
        ax.axhspan(0.90, 0.98, facecolor="none", edgecolor="#999999",
                   linestyle=":", linewidth=0.8, zorder=1)

        axin = inset_axes(ax, width="40%", height="32%", loc="center right", borderpad=1.4)
        draw(axin, metric, ["Synth", "3Di", "SS"], legend=False, small=True)
        axin.set_ylim(0.90, 0.98)
        axin.set_xlim(2, 53)
        axin.set_xticks(EPOCHS)
        axin.tick_params(labelsize=7)
        axin.axvline(DEPLOYED, color="#666666", linestyle="--", linewidth=0.8, zorder=1)
        axin.grid(True, linewidth=0.3, alpha=0.3)
        axin.set_facecolor("#ffffff")
        axin.set_title("detail: 0.90\u20130.98", fontsize=7.5, color="#444444", pad=3)
        for side in ("top", "right"):
            axin.spines[side].set_visible(False)
        ax.legend(frameon=False, fontsize=9, loc="center left", ncol=1)
    else:
        ax.legend(frameon=False, fontsize=9, loc="best", ncol=2)

    fig.tight_layout()
    fig.savefig(fname, dpi=220, bbox_inches="tight")
    return fig


PATHS = []
for _m, _lab, _lim, _fn in METRICS:
    _fig = build(_m, _lab, _lim, _fn)
    PATHS.append(_fn)
    plt.show()

print("wrote:", PATHS)


In [ ]:
# Chrome blocks the second and later files of a multi-file files.download, so
# the three PNGs go out as a single zip.
import zipfile

ZIP = "colab42_epoch_figures.zip"
with zipfile.ZipFile(ZIP, "w") as z:
    for p in PATHS:
        z.write(p)

try:
    from google.colab import files
    files.download(ZIP)
except ImportError:
    print("not in Colab - files are in the working directory:", PATHS)


## Where the files go

Unzip into `Latex_write_up/latex-template-cgv/fig/`, overwriting the three
existing PNGs. The filenames are already what `5_discussion.tex` expects, so no
LaTeX edit is needed - rebuild and Figures 5.3-5.5 pick up the new versions.

The repository also carries `colab_outputs/colab42_epoch_figures.py`, which does
the same thing but reads the CSV instead of embedded numbers. Use that one if
the ablation run is ever re-run, since it cannot drift from the data.
